# 09 — Does Image Enhancement Recover Fish Freshness Classification Under Dark Conditions?

**Untuk artikel JRI | Bahasa pengantar: Indonesia | Dataset: FFE**

### Apa pertanyaannya?

> Setelah foto mata ikan dibuat lebih gelap, apakah **Gamma Correction** atau **CLAHE** dapat memperbaiki prediksi tiga classifier yang sudah dilatih dengan Swin-Tiny?

**Yang dilakukan notebook:**

1. Menggunakan **checkpoint Swin-Tiny** dan classifier **FCN, Logistic Regression, LightGBM** dari notebook 02–04, **tanpa melatih ulang**.
2. Memilih **parameter Gamma dan CLAHE** berdasarkan *validation set* untuk setiap fold. Citra *test* tidak digunakan untuk memilih parameter.
3. Menguji pada **foto normal**, **gelap ringan**, dan **gelap berat**, memakai **ID foto dan fold identik** dengan notebook 08.
4. Menghitung Macro F1, akurasi, kesalahan *Not Fresh → Highly Fresh*, per spesies, dan **95% paired bootstrap CI**.
5. Menyimpan tabel dan gambar yang dapat digunakan sebagai dasar **Results and Discussion**.

**Perlu dicatat:** hasil notebook 08 telah dipakai untuk memilih topik penelitian ini. Jadi eksperimen 09 adalah **follow-up eksploratif**, bukan validasi eksternal atau uji konfirmatori yang sepenuhnya independen. Perubahan gambar bersifat sintetis. Jangan mengklaim model aman digunakan untuk menentukan keamanan konsumsi ikan.

**Urutan penggunaan:** Runtime → Change runtime type → **T4 GPU** → **Run all**. Tidak perlu mengulang notebook 00–08.

In [ ]:
%pip -q install "timm==1.0.31" "scikit-learn==1.6.1" "lightgbm>=4,<5" "joblib>=1.3" "opencv-python-headless>=4.8,<5" "pandas>=2,<3" "matplotlib>=3.7"

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys

drive.mount('/content/drive')
PROJECT = Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Workspace tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT']=str(PROJECT)
sys.path.insert(0,str(PROJECT/'02_notebooks'))
from ffe_core import (paths, ensure_dataset_local, load_manifest, load_split,
                      image_path, get_transform, extract_feature_checkpoint,
                      DeepOnlyNetFactory, predict_logits, metric_row,
                      CLASS_NAMES, test_group_integrity, seed_all)
P=paths(); seed_all(42)
print('Workspace:',PROJECT)

## 1. Konfigurasi penelitian

**Agar hasil sah untuk penulisan artikel, biarkan `FULL_TEST=True`.** Mode `FULL_TEST=False` hanya untuk mencoba apakah kode berjalan.

Parameter koreksi dipilih dari daftar kecil yang sudah ditetapkan: Gamma = 0,70 atau 0,85; CLAHE clipLimit = 1,5 atau 2,5 (tile 8×8). Nilai gamma di bawah satu **mencerahkan** gambar. CLAHE hanya diterapkan pada kanal kecerahan ruang warna LAB agar perubahan warna tidak berlebihan.

**Protokol:** untuk setiap fold, rata-rata Macro F1 pada *validation* gelap ringan dan gelap berat dari ketiga classifier menentukan satu parameter per metode. Parameter terpilih dipakai **tanpa diubah** pada test. Performa pada foto normal juga diuji, tetapi **tidak digunakan untuk seleksi**.

In [ ]:
import json, hashlib, gc, time, warnings, math
import numpy as np, pandas as pd, cv2, torch, timm, joblib
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (f1_score, confusion_matrix, recall_score)
from IPython.display import display

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FOLDS=[1,2,3,4,5]
MODELS=['deep_fcn','deep_lr','deep_lgbm']
MODEL_LABEL={'deep_fcn':'Swin + FCN','deep_lr':'Swin + Logistic Regression',
             'deep_lgbm':'Swin + LightGBM'}
CONDITION_NAMES=['original','dark_mild','dark_strong']
METHODS=['none','gamma','clahe']
CANDIDATES={
  'gamma':[0.70,0.85],
  'clahe':[1.5,2.5],
}
FULL_TEST=True
SMOKE_N=48
RESUME=True
BATCH_SIZE=24 if DEVICE.type=='cuda' else 8
N_BOOT=1000
SEED=20261109

BASE08=P['metrics']/'robustness_jri'
BASE08_PRED=BASE08/'predictions'
BASE08_CONFIG=BASE08/'experiment_config.json'
OUT_NAME='enhancement_jri' if FULL_TEST else 'enhancement_jri_smoke'
METRICS_DIR=P['metrics']/OUT_NAME
FIGURES_DIR=P['figures']/OUT_NAME
PRED_DIR=METRICS_DIR/'predictions'
for d in (METRICS_DIR,FIGURES_DIR,PRED_DIR): d.mkdir(parents=True,exist_ok=True)

SETUP={
 'methods':METHODS,'conditions':CONDITION_NAMES,'candidates':CANDIDATES,
 'folds':FOLDS,'models':MODELS,'seed':SEED,'full_test':FULL_TEST,
 'smoke_n':SMOKE_N,'selection':'mean val macro-F1 over three classifiers and two dark conditions',
 'normal_safety':'report only, not used for hyperparameter selection',
 'augmentation_order':'darkening (notebook08) followed by image enhancement'
}
RUN_HASH=hashlib.sha256(json.dumps(SETUP,sort_keys=True).encode()).hexdigest()[:16]
print('GPU:',DEVICE,'RUN_HASH:',RUN_HASH,'output:',METRICS_DIR)
if not FULL_TEST:
    print('SMOKE MODE: jangan gunakan hasil ini sebagai angka final artikel.')

## 2. Memeriksa artefak penelitian sebelumnya

Notebook ini **tidak menimpa data atau prediksi notebook 08**. Prediksi original/dark dari notebook 08 dipakai sebagai *baseline*, setelah dipastikan semua ID sampel dan label cocok. Jika ada berkas hilang atau eksperimen 08 belum `FULL`, kode berhenti dengan pesan kesalahan.

In [ ]:
assert BASE08_CONFIG.exists(), 'Notebook 08 belum selesai: experiment_config.json tidak ditemukan.'
config08=json.loads(BASE08_CONFIG.read_text(encoding='utf-8'))
assert config08.get('full_test') is True, 'Notebook 08 harus sudah dieksekusi FULL_TEST=True.'
assert config08.get('n_unique_images')==4390, 'Notebook 08 tidak menggunakan 4390 foto.'
assert config08.get('folds')==FOLDS, 'Daftar fold berbeda dengan notebook 08.'
assert config08.get('models')==MODELS, 'Model berbeda dari notebook 08.'
CONF08={c['name']:c for c in config08['conditions']}
assert all(x in CONF08 for x in CONDITION_NAMES)
assert CONF08['dark_mild']['kind']=='dark' and np.isclose(CONF08['dark_mild']['factor'],.8)
assert CONF08['dark_strong']['kind']=='dark' and np.isclose(CONF08['dark_strong']['factor'],.6)
assert CONF08['original']['kind']=='original'
manifest=load_manifest()
assert len(manifest)==4390 and test_group_integrity()
required=[]
for fold in FOLDS:
  required += [P['checkpoints']/f'swin_fold_{fold:02d}_best.pt',
       P['models']/f'deep_scaler_fold_{fold:02d}.joblib',
       P['models']/f'deep_lr_fold_{fold:02d}.joblib',
       P['models']/f'deep_lgbm_fold_{fold:02d}.joblib',
       P['models']/f'fold_{fold:02d}'/'deep_fcn.pt']
  required += [BASE08_PRED/f'fold_{fold:02d}'/f'{c}.csv' for c in CONDITION_NAMES]
missing=[str(p) for p in required if not p.exists()]
assert not missing,'Artefak yang belum ada:\n'+'\n'.join(missing[:15])
ensure_dataset_local()
print(f'OK — {len(required)} model dan baseline tersedia; {len(manifest)} foto dalam manifest.')

## 3. Fungsi mempergelap, memperbaiki citra, dan mengklasifikasikan

Pada penelitian ini gambar diproses dalam urutan **citra FFE asli → simulasi foto gelap → perbaikan → Swin-Tiny → classifier**. Gambar asli tidak diubah di Drive.

Koreksi Gamma memakai *lookup table* yang tetap. CLAHE bekerja pada kanal `L` dari LAB 8-bit. Parameter dipilih di validation, **tanpa melihat hasil test**.

In [ ]:
TRANSFORM=get_transform(False)

def perturb_image(rgb_pil, condition):
    kind=condition['kind']
    if kind=='original': return rgb_pil.copy()
    assert kind=='dark',f'Kondisi tidak didukung notebook 09: {condition}'
    x=np.asarray(rgb_pil,dtype=np.uint8)
    y=np.clip(np.rint(x.astype(np.float32)*float(condition['factor'])),0,255).astype(np.uint8)
    return Image.fromarray(y,'RGB')

def enhance_image(rgb_pil,method,param=None):
    if method=='none':return rgb_pil.copy()
    x=np.asarray(rgb_pil,dtype=np.uint8)
    if method=='gamma':
        gamma=float(param)
        assert 0.1<gamma<=1.0
        lut=np.clip(np.rint(255.0*(np.arange(256,dtype=np.float32)/255.0)**gamma),0,255).astype(np.uint8)
        result=cv2.LUT(x,lut)
    elif method=='clahe':
        lab=cv2.cvtColor(x,cv2.COLOR_RGB2LAB)
        l,a,b=cv2.split(lab)
        clahe=cv2.createCLAHE(clipLimit=float(param),tileGridSize=(8,8))
        lab_new=cv2.merge((clahe.apply(l),a,b))
        result=cv2.cvtColor(lab_new,cv2.COLOR_LAB2RGB)
    else:raise ValueError(f'Metode tidak dikenal: {method}')
    return Image.fromarray(result,'RGB')

# Uji singkat algoritma pengolahan citra (tanpa memakai data test)
_demo=np.full((60,60,3),80,dtype=np.uint8)
for m,p in [('none',None),('gamma',.70),('clahe',2.5)]:
    out=np.array(enhance_image(Image.fromarray(_demo,'RGB'),m,p))
    assert out.shape==_demo.shape and out.dtype==np.uint8
assert np.array(enhance_image(Image.fromarray(_demo,'RGB'),'gamma',.70)).mean()>_demo.mean()
print('OK — fungsi Gamma dan CLAHE lulus uji dasar.')

In [ ]:
def extract_embeddings(df,backbone,condition,method='none',param=None):
    """Ekstraksi ulang embedding citra yang sudah diproses; tidak melatih backbone."""
    assert len(df)>0
    output=[]
    backbone.eval()
    for start in range(0,len(df),BATCH_SIZE):
        block=df.iloc[start:start+BATCH_SIZE]
        tensors=[]
        for row in block.itertuples(index=False):
            with Image.open(image_path(row.rel_path)) as image:
                original=image.convert('RGB')
                altered=perturb_image(original,condition)
                corrected=enhance_image(altered,method,param)
                tensors.append(TRANSFORM(corrected))
        batch=torch.stack(tensors).to(DEVICE)
        with torch.inference_mode():
            with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
                z=backbone(batch)
        output.append(z.float().cpu().numpy())
    X=np.concatenate(output,axis=0).astype(np.float32)
    assert X.shape==(len(df),768) and np.isfinite(X).all()
    return X

def load_classifiers(fold):
    scaler=joblib.load(P['models']/f'deep_scaler_fold_{fold:02d}.joblib')
    ck=torch.load(P['models']/f'fold_{fold:02d}'/'deep_fcn.pt',
                  map_location='cpu',weights_only=False)
    fcn=DeepOnlyNetFactory.create().to(DEVICE)
    fcn.load_state_dict(ck['state_dict']);fcn.eval()
    lr=joblib.load(P['models']/f'deep_lr_fold_{fold:02d}.joblib')
    lgbm=joblib.load(P['models']/f'deep_lgbm_fold_{fold:02d}.joblib')
    assert np.array_equal(lr.classes_,[0,1,2])
    assert np.array_equal(lgbm.classes_,[0,1,2])
    return fcn,lr,lgbm,scaler

def classify(X,df,fold,condition,method,param,models):
    fcn,lr,lgbm,scaler=models
    x_scaled=scaler.transform(X).astype(np.float32)
    logits=predict_logits(fcn,x_scaled,bs=512,device=DEVICE)
    prob_fcn=torch.softmax(torch.from_numpy(logits),dim=1).numpy()
    probs={'deep_fcn':prob_fcn,'deep_lr':lr.predict_proba(X),
           'deep_lgbm':lgbm.predict_proba(X)}
    result=df[['sample_id','species','freshness','label']].copy().rename(columns={'label':'y_true'})
    result.insert(0,'fold',int(fold))
    result.insert(1,'condition',condition)
    result.insert(2,'method',method)
    result['param']=np.nan if param is None else float(param)
    result['run_hash']=RUN_HASH
    for model,p in probs.items():
        assert p.shape==(len(result),3) and np.isfinite(p).all()
        assert np.allclose(p.sum(axis=1),1,atol=1e-3)
        result[f'{model}_pred']=np.argmax(p,axis=1).astype(int)
        for k in range(3):result[f'{model}_p{k}']=p[:,k]
    return result

## 4. Memilih parameter menggunakan validation set saja

**Tidak ada data test pada tahap ini.** Untuk tiap fold, Gamma 0,70 vs 0,85 dan CLAHE 1,5 vs 2,5 diuji pada validation set yang digelapkan. Skor seleksi adalah rerata Macro F1 dari **3 classifier × 2 tingkat gelap**. Pilihan tersebut disimpan sehingga dapat diaudit.

Jika `RESUME=True`, notebook dapat menggunakan hasil parameter validation yang cocok dengan konfigurasi, sehingga tidak perlu mengulangi ekstraksi embedding validation bila runtime Colab terputus.

In [ ]:
def macro_f1(y,p):
    return f1_score(y,p,labels=[0,1,2],average='macro',zero_division=0)

def fold_input(fold):
    val=load_split(fold,'val').reset_index(drop=True)
    test=load_split(fold,'test').reset_index(drop=True)
    assert set(val.sample_id).isdisjoint(set(test.sample_id))
    if not FULL_TEST:
        val=val.sample(n=min(SMOKE_N,len(val)),random_state=SEED+fold).reset_index(drop=True)
        test=test.sample(n=min(SMOKE_N,len(test)),random_state=SEED+100+fold).reset_index(drop=True)
    return val,test

def ids_hash(df):
    return hashlib.sha256('|'.join(df.sample_id.astype(str).tolist()).encode()).hexdigest()[:16]

def read_baseline08(fold,condition,test):
    fn=BASE08_PRED/f'fold_{fold:02d}'/f'{condition}.csv'
    old=pd.read_csv(fn,dtype={'sample_id':str,'config_hash':str})
    assert old.sample_id.is_unique and (old.condition==condition).all()
    assert (old['config_hash'].astype(str)==str(config08['config_hash'])).all()
    old=old.set_index('sample_id')
    ids=test.sample_id.astype(str).tolist()
    assert set(ids).issubset(set(old.index))
    old=old.loc[ids].reset_index()
    assert np.array_equal(old.y_true.to_numpy(dtype=int),test.label.to_numpy(dtype=int))
    assert (old.species.to_numpy()==test.species.to_numpy()).all()
    old['method']='none';old['param']=np.nan;old['run_hash']=RUN_HASH
    return old

def choose_validation_parameters(fold,val,backbone,models):
    local_dir=METRICS_DIR/'validation';local_dir.mkdir(parents=True,exist_ok=True)
    mfile=local_dir/f'fold_{fold:02d}_candidate_metrics.csv'
    sfile=local_dir/f'fold_{fold:02d}_selection.csv'
    if RESUME and mfile.exists() and sfile.exists():
        cm=pd.read_csv(mfile); sel=pd.read_csv(sfile)
        if (len(cm)==30 and len(sel)==2 and
            cm.run_hash.eq(RUN_HASH).all() and sel.run_hash.eq(RUN_HASH).all() and
            cm.val_ids_hash.eq(ids_hash(val)).all() and sel.val_ids_hash.eq(ids_hash(val)).all() and
            set(sel.method)=={'gamma','clahe'}):
            print(' [cache] Pilihan parameter validation tersedia')
            return cm,sel
        raise RuntimeError(f'Cache validation tidak cocok: {mfile}. Hapus kedua file untuk mengulang.')
    rows=[]
    for method,options in [('none',[None]),('gamma',CANDIDATES['gamma']),('clahe',CANDIDATES['clahe'])]:
        for par in options:
            for cname in ('dark_mild','dark_strong'):
                X=extract_embeddings(val,backbone,CONF08[cname],method,par)
                pred=classify(X,val,fold,cname,method,par,models)
                for model in MODELS:
                    rows.append({'fold':fold,'condition':cname,'method':method,
                        'param':np.nan if par is None else float(par),'model':model,
                        'macro_f1':macro_f1(val.label.to_numpy(dtype=int),pred[f'{model}_pred'].to_numpy(dtype=int)),
                        'n_val':len(val),'val_ids_hash':ids_hash(val),'run_hash':RUN_HASH})
                del X,pred
            print(f' validation fold={fold}: {method} param={par} OK')
    cm=pd.DataFrame(rows)
    assert len(cm)==30
    selections=[]
    for method in ('gamma','clahe'):
        q=cm[cm.method==method].groupby('param',as_index=False).macro_f1.mean()
        q=q.sort_values(['macro_f1','param'],ascending=[False,True]).iloc[0]
        selections.append({'fold':fold,'method':method,'param':float(q['param']),
                  'validation_mean_f1':float(q['macro_f1']),
                  'n_val':len(val),'val_ids_hash':ids_hash(val),'run_hash':RUN_HASH})
    sel=pd.DataFrame(selections)
    cm.to_csv(mfile,index=False);sel.to_csv(sfile,index=False)
    return cm,sel

## 5. Eksperimen test: tiga kondisi × dua teknik perbaikan

Prediksi original/dark **tanpa perbaikan** diambil dari notebook 08. Prediksi **Gamma** dan **CLAHE** dihitung sekarang untuk sampel identik. Sebelum memakai baseline lama, notebook melakukan **pemeriksaan kesamaan prediksi** pada original tiap fold; jika terdapat ketidakcocokan, proses dihentikan agar hasil tidak digabung secara tidak sah.

Setiap kondisi dan metode disimpan dalam CSV terpisah untuk memudahkan melanjutkan pekerjaan apabila sesi GPU terputus. Data test hanya dipakai untuk evaluasi setelah parameter dari validation ditetapkan.

In [ ]:
all_val_metrics=[];all_selections=[];all_test=[]
t0=time.time()
for fold in FOLDS:
    print(f'\n======== FOLD {fold}/5 ========')
    val,test=fold_input(fold)
    backbone=extract_feature_checkpoint(P['checkpoints']/f'swin_fold_{fold:02d}_best.pt',DEVICE)
    models=load_classifiers(fold)
    try:
        cm,selection=choose_validation_parameters(fold,val,backbone,models)
        all_val_metrics.append(cm);all_selections.append(selection)
        print(' parameter dipilih:',selection[['method','param','validation_mean_f1']].round(4).to_dict('records'))
        baseline={c:read_baseline08(fold,c,test) for c in CONDITION_NAMES}
        # Verifikasi keras bahwa backbone + preprocessing original sama dengan baseline 08.
        X_original=extract_embeddings(test,backbone,CONF08['original'],'none',None)
        fresh_original=classify(X_original,test,fold,'original','none',None,models)
        for model in MODELS:
            a=fresh_original[f'{model}_pred'].to_numpy(dtype=int)
            b=baseline['original'][f'{model}_pred'].to_numpy(dtype=int)
            mismatches=int(np.sum(a!=b))
            print(f'  original parity {model}: {mismatches}/{len(test)} berbeda')
            if mismatches:
                raise RuntimeError(f'PARITY GAGAL model={model}, fold={fold}. Jangan gunakan gabungan hasil. Cek versi timm/PIL/preprocessing.')
        del X_original,fresh_original
        all_test.extend(baseline.values())
        for method in ('gamma','clahe'):
            par=float(selection.set_index('method').loc[method,'param'])
            for cname in CONDITION_NAMES:
                out_dir=PRED_DIR/f'fold_{fold:02d}';out_dir.mkdir(parents=True,exist_ok=True)
                filename=out_dir/f'{cname}__{method}.csv'
                if RESUME and filename.exists():
                    pred=pd.read_csv(filename,dtype={'sample_id':str,'run_hash':str})
                    good=(len(pred)==len(test) and pred.sample_id.tolist()==test.sample_id.astype(str).tolist() and
                        pred.run_hash.eq(RUN_HASH).all() and pred.method.eq(method).all() and
                        pred.condition.eq(cname).all() and np.allclose(pred.param,par))
                    if not good:raise RuntimeError(f'Cache prediksi tidak cocok: {filename}. Hapus file tersebut untuk mengulang.')
                    print(' [cache]',filename.name)
                else:
                    X=extract_embeddings(test,backbone,CONF08[cname],method,par)
                    pred=classify(X,test,fold,cname,method,par,models)
                    pred.to_csv(filename,index=False)
                    del X
                    print(' [saved]',filename.name,len(pred))
                all_test.append(pred)
    finally:
        del backbone,models
        gc.collect()
        if DEVICE.type=='cuda':torch.cuda.empty_cache()

V_METRICS=pd.concat(all_val_metrics,ignore_index=True)
SELECTIONS=pd.concat(all_selections,ignore_index=True)
PREDS=pd.concat(all_test,ignore_index=True)
V_METRICS.to_csv(METRICS_DIR/'validation_candidate_metrics.csv',index=False)
SELECTIONS.to_csv(METRICS_DIR/'validation_selected_params.csv',index=False)
assert PREDS.sample_id.nunique()==(4390 if FULL_TEST else len(pd.concat([fold_input(f)[1] for f in FOLDS])))
assert len(PREDS)==PREDS.sample_id.nunique()*len(CONDITION_NAMES)*len(METHODS)
assert PREDS.groupby(['condition','method']).sample_id.nunique().eq(PREDS.sample_id.nunique()).all()
print('\nSELESAI INFERENSI — jumlah foto unik',PREDS.sample_id.nunique())
display(SELECTIONS[['fold','method','param','validation_mean_f1']].round(4))

## 6. Tabel performa akhir (test)

**Baca tabel begini:** untuk setiap kondisi, bandingkan `none` dengan `gamma` dan `clahe` pada classifier yang sama. Jika Macro F1 naik, perbaikan gambar membantu pada kondisi tersebut. Jangan langsung menyebut metode terbaik tanpa melihat ketidakpastian dan kerugian pada foto asli.

In [ ]:
def evaluate(group,model):
    y=group.y_true.to_numpy(dtype=int)
    p=group[f'{model}_pred'].to_numpy(dtype=int)
    met=metric_row(y,p)
    cm=confusion_matrix(y,p,labels=[0,1,2])
    met['n']=len(y)
    met['not_fresh_recall']=recall_score(y,p,labels=[2],average=None,zero_division=0)[0]
    met['not_fresh_as_highly_fresh_n']=int(cm[2,0])
    met['not_fresh_as_highly_fresh_rate']=cm[2,0]/max(int(cm[2].sum()),1)
    return met

pooled=[];folded=[];species=[];classes=[]
for (condition,method),g in PREDS.groupby(['condition','method'],sort=False):
    for model in MODELS:
        pooled.append({'condition':condition,'method':method,'model':model,**evaluate(g,model)})
for (fold,condition,method),g in PREDS.groupby(['fold','condition','method'],sort=False):
    for model in MODELS:folded.append({'fold':fold,'condition':condition,'method':method,'model':model,**evaluate(g,model)})
for (condition,method,sp),g in PREDS.groupby(['condition','method','species'],sort=False):
    for model in MODELS:species.append({'condition':condition,'method':method,'species':sp,'model':model,**evaluate(g,model)})
for (condition,method),g in PREDS.groupby(['condition','method'],sort=False):
    y=g.y_true.to_numpy(dtype=int)
    for model in MODELS:
        p=g[f'{model}_pred'].to_numpy(dtype=int)
        for k,name in enumerate(CLASS_NAMES):
            mask=(y==k)
            classes.append({'condition':condition,'method':method,'model':model,
                            'class':name,'n':int(mask.sum()),
                            'recall':float((p[mask]==k).mean()) if mask.any() else float('nan')})
POOLED=pd.DataFrame(pooled);FOLDED=pd.DataFrame(folded)
SPECIES=pd.DataFrame(species);CLASSES=pd.DataFrame(classes)
for name,df in [('pooled_metrics.csv',POOLED),('fold_metrics.csv',FOLDED),
                ('per_species.csv',SPECIES),('per_class_recall.csv',CLASSES)]:
    df.to_csv(METRICS_DIR/name,index=False)
print('Macro F1 tertinggi = lebih baik, belum tentu berbeda signifikan:')
display(POOLED.pivot_table(index=['condition','method'],columns='model',values='macro_f1').round(4))
print('Kesalahan Not Fresh → Highly Fresh (jumlah foto):')
display(POOLED.pivot_table(index=['condition','method'],columns='model',values='not_fresh_as_highly_fresh_n').astype(int))

## 7. Uji perbandingan berpasangan dan interval kepercayaan 95%

Untuk setiap kondisi dan classifier, hitung **Macro F1(corrected) − Macro F1(uncorrected)** pada foto uji yang sama. Nilai positif berarti terjadi perbaikan. Interval bootstrap dibangun dengan memilih ulang indeks foto **secara berpasangan** agar foto pada kedua metode tetap identik.

**Interpretasi:** interval yang tidak mencakup nol adalah bukti indikatif untuk perbandingan itu; penelitian ini memiliki beberapa perbandingan dan bootstrap belum mengakomodasi keterkaitan antarindividu ikan/sesi pengambilan gambar. Jangan membuat klaim signifikansi global atau kausalitas dunia nyata.

In [ ]:
def macro_f1_fast(y,p,k=3):
    cm=np.bincount(y*k+p,minlength=k*k).reshape(k,k)
    tp=np.diag(cm)
    denominator=cm.sum(axis=0)+cm.sum(axis=1)
    return float(np.mean(np.divide(2*tp,denominator,out=np.zeros(k,float),where=denominator>0)))

def paired_ci(y,corrected,uncorrected,n_boot=N_BOOT,seed=SEED):
    y=np.asarray(y,dtype=int);a=np.asarray(corrected,dtype=int);b=np.asarray(uncorrected,dtype=int)
    assert len(y)==len(a)==len(b)
    delta=macro_f1_fast(y,a)-macro_f1_fast(y,b)
    rng=np.random.default_rng(seed)
    draws=np.empty(n_boot,float)
    for i in range(n_boot):
        indices=rng.integers(0,len(y),len(y))
        draws[i]=macro_f1_fast(y[indices],a[indices])-macro_f1_fast(y[indices],b[indices])
    return delta,np.quantile(draws,[.025,.975])

comp=[]
for cname in CONDITION_NAMES:
    subset=PREDS[PREDS.condition==cname]
    original=subset[subset.method=='none'].set_index('sample_id')
    assert original.index.is_unique
    for method in ('gamma','clahe'):
        corrected=subset[subset.method==method].set_index('sample_id')
        assert corrected.index.is_unique and set(original.index)==set(corrected.index)
        corrected=corrected.loc[original.index]
        y=original.y_true.to_numpy(dtype=int)
        assert np.array_equal(y,corrected.y_true.to_numpy(dtype=int))
        for model in MODELS:
            seed=SEED+int(hashlib.sha256(f'{cname}:{method}:{model}'.encode()).hexdigest()[:8],16)
            gain,ci=paired_ci(y,corrected[f'{model}_pred'].to_numpy(dtype=int),
                original[f'{model}_pred'].to_numpy(dtype=int),seed=seed)
            cor_err=int(((y==2)&(corrected[f'{model}_pred'].to_numpy(dtype=int)==0)).sum())
            base_err=int(((y==2)&(original[f'{model}_pred'].to_numpy(dtype=int)==0)).sum())
            comp.append({'condition':cname,'method':method,'model':model,
                  'gain_macro_f1':gain,'ci95_low':ci[0],'ci95_high':ci[1],
                  'gain_points':100*gain,'n':len(y),
                  'critical_error_corrected_n':cor_err,'critical_error_baseline_n':base_err,
                  'critical_error_change_n':cor_err-base_err})
GAINS=pd.DataFrame(comp)
GAINS.to_csv(METRICS_DIR/'paired_enhancement_gain_bootstrap.csv',index=False)
print('POSITIF pada gain_macro_f1 berarti koreksi membantu:')
display(GAINS[['condition','method','model','gain_macro_f1','ci95_low','ci95_high','critical_error_change_n']].round(4))

## 8. Grafik dan contoh foto

Grafik memperlihatkan skor **tanpa koreksi**, **Gamma Correction**, dan **CLAHE**. Foto contoh diambil dari bagian **validation** fold 1, bukan dipilih berdasarkan keberhasilan pada test.

In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
colors={'none':'#64748b','gamma':'#2563eb','clahe':'#16a34a'}
fig,axs=plt.subplots(1,3,figsize=(15,4.2),sharey=True)
for ax,model in zip(axs,MODELS):
    for method in METHODS:
        q=POOLED[(POOLED.model==model)&(POOLED.method==method)].set_index('condition').loc[CONDITION_NAMES]
        ax.plot(range(3),q.macro_f1*100,marker='o',linewidth=2,
                label={'none':'No correction','gamma':'Gamma','clahe':'CLAHE'}[method],color=colors[method])
    ax.set_title(MODEL_LABEL[model]);ax.set_xticks(range(3),['Normal','Dark mild','Dark strong'])
    ax.grid(axis='y',alpha=.25)
axs[0].set_ylabel('Macro F1 (%)');axs[-1].legend(loc='best')
fig.suptitle('Effect of test-time image enhancement on Swin-Tiny classifiers',y=1.02)
fig.tight_layout();fig.savefig(FIGURES_DIR/'macro_f1_by_enhancement.png',dpi=200,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(figsize=(10,5))
dark=GAINS[GAINS.condition=='dark_strong'].copy()
pos=np.arange(len(MODELS));width=.35
for idx,method in enumerate(['gamma','clahe']):
    q=dark[dark.method==method].set_index('model').loc[MODELS]
    points=q.gain_points.to_numpy(dtype=float)
    lower=100*(q.gain_macro_f1-q.ci95_low).to_numpy(dtype=float)
    upper=100*(q.ci95_high-q.gain_macro_f1).to_numpy(dtype=float)
    ax.bar(pos+(idx-.5)*width,points,width,color=colors[method],label=method.upper())
    ax.errorbar(pos+(idx-.5)*width,points,yerr=[lower,upper],fmt='none',ecolor='black',capsize=3)
ax.axhline(0,color='black',linewidth=1)
ax.set_xticks(pos,[MODEL_LABEL[m] for m in MODELS])
ax.set_ylabel('Macro F1 improvement (percentage points)')
ax.set_title('Dark strong: paired gain over no correction (95% image-bootstrap CI)')
ax.legend();ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(FIGURES_DIR/'paired_gain_dark_strong.png',dpi=200,bbox_inches='tight');plt.show()

fig,axs=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,cname in zip(axs,['dark_mild','dark_strong']):
    q=POOLED[POOLED.condition==cname]
    x=np.arange(len(MODELS));width=.26
    for j,method in enumerate(METHODS):
        sub=q[q.method==method].set_index('model').loc[MODELS]
        ax.bar(x+(j-1)*width,sub.not_fresh_as_highly_fresh_rate*100,width,
               label=method,color=colors[method])
    ax.set_xticks(x,['FCN','LR','LightGBM']);ax.set_title(cname.replace('_',' ').title());ax.grid(axis='y',alpha=.2)
axs[0].set_ylabel('Not Fresh predicted Highly Fresh (%) of Not Fresh')
axs[-1].legend()
fig.tight_layout();fig.savefig(FIGURES_DIR/'critical_error_rate.png',dpi=200,bbox_inches='tight');plt.show()

# Contoh visual: gunakan validation fold 1, bukan test. Tidak ada pemilihan berdasarkan label atau prediksi.
val01,_=fold_input(1)
row=val01.iloc[0]
with Image.open(image_path(row.rel_path)) as sample:
    raw=sample.convert('RGB');dark=perturb_image(raw,CONF08['dark_strong'])
params=SELECTIONS[(SELECTIONS.fold==1)].set_index('method').param
imgs=[raw,dark,enhance_image(dark,'gamma',float(params['gamma'])),
      enhance_image(dark,'clahe',float(params['clahe']))]
fig,axs=plt.subplots(1,4,figsize=(12,3.5))
for ax,im,title in zip(axs,imgs,['Original','Dark strong','Gamma corrected','CLAHE corrected']):
    ax.imshow(im);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(FIGURES_DIR/'validation_image_example.png',dpi=200,bbox_inches='tight');plt.show()

## 9. Kesimpulan otomatis dan batasan untuk manuskrip

Hasil di bawah adalah **deskripsi otomatis**, bukan bukti bahwa perbaikan gambar selalu efektif. Kegiatan ini memanfaatkan model serta data test yang sudah dianalisis pada notebook 08; validitas klaim eksternal memerlukan pengujian pada individu ikan, sesi kamera, atau dataset independen.

In [ ]:
REPORT=['# Experiment 09 — Image Enhancement and Swin-Tiny Fish Freshness Classification',
        '',f'- Status: {"FULL" if FULL_TEST else "SMOKE / NOT FOR PAPER"}',
        f'- Unik: {PREDS.sample_id.nunique()} citra, 5 folds, 3 classifier',
        '- Enhancement: Gamma Correction and CLAHE (LAB L channel)',
        '- Hyperparameter: selected per outer fold using validation set only',
        '- Test inputs: identical sample IDs to notebook 08; verified original-prediction parity',
        '- Dark synthetic brightness factors: 0.8 and 0.6',
        f'- Configuration hash: {RUN_HASH}',
        '', '## Pilihan parameter pada validation',
        '',SELECTIONS[['fold','method','param','validation_mean_f1']].round(5).to_markdown(index=False),
        '', '## Makro F1 pooled out-of-fold pada test',
        '',POOLED[['condition','method','model','macro_f1','accuracy','not_fresh_as_highly_fresh_n']].round(5).to_markdown(index=False),
        '', '## Perubahan Macro F1 karena enhancement',
        '',GAINS[['condition','method','model','gain_macro_f1','ci95_low','ci95_high','critical_error_change_n']].round(5).to_markdown(index=False),
        '', '## Keterbatasan dan integritas penelitian',
        '- Enhancement berangkat dari pengetahuan hasil eksperimen 08; follow-up ini bersifat eksploratif.',
        '- Test hanya foto FFE dengan degradasi sintetis, bukan foto baru independen dari pasar.',
        '- SHA256-grouped cross-validation tidak menjamin independensi per individu ikan atau sesi pengambilan.',
        '- Paired image-bootstrap tidak mengakomodasi kemungkinan korelasi ikan/sesi.',
        '- Banyak perbandingan dilakukan. Jangan mengklaim signifikansi menyeluruh dari satu CI.',
        '- Label FFE berdasarkan kategori hari penyimpanan; bukan sertifikasi keamanan pangan.',
        '- CLAHE/Gamma mungkin merusak tampilan normal; efek pada kondisi normal ikut dilaporkan.',
        '- Pemilihan parameter memakai validation tiap fold dan bukan data test, tetapi validation sudah digunakan saat membangun model asli.',
        '', '## Output',
        '- metrics: validation_candidate_metrics.csv, validation_selected_params.csv, pooled_metrics.csv, fold_metrics.csv, per_species.csv, per_class_recall.csv, paired_enhancement_gain_bootstrap.csv, prediction CSVs',
        '- figures: macro_f1_by_enhancement.png, paired_gain_dark_strong.png, critical_error_rate.png, validation_image_example.png']
(METRICS_DIR/'JRI_enhancement_findings.md').write_text('\n'.join(REPORT),encoding='utf-8')
metadata={**SETUP,'config_hash':RUN_HASH,'baseline08_hash':config08['config_hash'],
          'device':str(DEVICE),'torch_version':torch.__version__,'timm_version':timm.__version__,
          'unique_test_images':int(PREDS.sample_id.nunique()),'rows':int(len(PREDS)),
          'data_independence':'fish/session independence unverified'}
(METRICS_DIR/'experiment_config.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('\nNotebook 09 selesai. Hasil ada di:')
print(METRICS_DIR)
print(FIGURES_DIR)
print('\nRingkasan temuan:')
for r in GAINS[GAINS.condition=='dark_strong'].itertuples(index=False):
    print(f'{MODEL_LABEL[r.model]} / {r.method}: {r.gain_points:+.2f} poin Macro F1, CI [{100*r.ci95_low:+.2f},{100*r.ci95_high:+.2f}]')